In [ ]:
###
# セットアップ
###

from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")
except Exception:
    ROOT_PATH = Path.cwd()

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

LOCAL_DATA_DIR = ROOT_PATH / "data" / "cats_vs_dogs"
if not LOCAL_DATA_DIR.exists():
    fallback = Path("/content/data/cats_vs_dogs")
    if fallback.exists():
        LOCAL_DATA_DIR = fallback

print("ROOT_PATH:", ROOT_PATH)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)


In [ ]:
###
# 1. 必要なライブラリのインポート
###

import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")


In [ ]:
###
# 2. 評価用Dataset / DataLoader
###

_FRAC_MAP = {"small": 0.2, "medium": 0.5, "large": 1.0}

class CDDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.data_dir, row["filepath"])).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(row["label"])


EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def get_dc_dataloaders(data_dir, data_size="small", batch_size=32):
    frac = _FRAC_MAP.get(data_size)
    if frac is None:
        raise ValueError("data_size must be one of small, medium, large")

    df = pd.read_csv(os.path.join(data_dir, "labels.csv"))

    def make_loader(split, shuffle):
        split_df = df[df["split"] == split].sample(frac=frac, random_state=61)
        dataset = CDDataset(split_df, data_dir, transform=EVAL_TRANSFORM)
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=2,
            pin_memory=torch.cuda.is_available(),
        )

    train_loader = make_loader("train", True)
    val_loader = make_loader("val", False)
    test_loader = make_loader("test", False)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_dc_dataloaders(
    data_dir=LOCAL_DATA_DIR,
    data_size="small",
    batch_size=32,
)

print("test batches:", len(test_loader))


In [ ]:
###
# 3. CNNを読み込む
###

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(in_features=128 * 8 * 8, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=2)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = self.conv4(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x


model = SimpleCNN().to(device)

candidate_paths = [
    ROOT_PATH / "models" / "11_data_augmentation.pth",
    ROOT_PATH / "models" / "06_cnn.pth",
]

save_path = None
for candidate in candidate_paths:
    if candidate.exists():
        save_path = candidate
        break

if save_path is None:
    raise FileNotFoundError("model checkpoint not found")

model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

print("loaded:", save_path)


In [ ]:
###
# 4. 混同行列と精度を確認する
###

label_names = {0: "cat", 1: "dog"}
confusion = torch.zeros(2, 2, dtype=torch.int64)
correct = 0
total = 0
mistakes = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        for image, label, pred in zip(images.cpu(), labels.cpu(), preds.cpu()):
            confusion[label, pred] += 1
            if label != pred and len(mistakes) < 12:
                mistakes.append((image, int(label), int(pred)))

accuracy = 100 * correct / total
print(f"test accuracy: {accuracy:.2f}%")
print("confusion matrix:\n", confusion)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(confusion.numpy(), cmap="Blues")
ax.set_xticks([0, 1], [label_names[0], label_names[1]])
ax.set_yticks([0, 1], [label_names[0], label_names[1]])
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(j, i, int(confusion[i, j]), ha="center", va="center", color="black")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
###
# 5. 誤分類例を確認する
###

def denormalize(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(-1, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(-1, 1, 1)
    image = image_tensor.clone() * std + mean
    return image.clamp(0, 1)


if not mistakes:
    print("誤分類は見つかりませんでした")
else:
    n_show = min(len(mistakes), 8)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()

    for ax in axes[n_show:]:
        ax.axis("off")

    for ax, (image, label, pred) in zip(axes, mistakes[:n_show]):
        ax.imshow(denormalize(image).permute(1, 2, 0))
        ax.set_title(f"true: {label_names[label]}\npred: {label_names[pred]}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()
